# 1. Temperature Dataset

In [1]:
import time
from datetime import datetime
import glob
import pandas as pd
import requests

In [2]:
# 1. Define coordinates for Rwanda's provinces/markets
# Example coordinates for the provinces are used here based on your dataset.
# You can replace these with the actual lat/lon coordinates of your markets.
locations = [
    {"name": "Kigali City", "lat": -1.9441, "lon": 30.0619},
    {"name": "Eastern Province", "lat": -1.95, "lon": 30.45},
    {"name": "Western Province", "lat": -2.15, "lon": 29.35},
    {"name": "Northern Province", "lat": -1.65, "lon": 29.85},
    {"name": "Southern Province", "lat": -2.45, "lon": 29.75},
]

# 2. Set the time range (matching your 2020–2026 price data)
START_DATE = "20200101"
END_DATE = "20261231"

# Target parameters: 
# T2M = Temperature at 2 Meters Daily Average (°C)
# PRECTOTCORR = Precipitation Corrected (mm/day)
# GWETPROF = Profile Soil Moisture (0 to 1 scale)
PARAMETERS = "T2M,PRECTOTCORR,GWETPROF"

# NASA POWER API base URL (daily resolution, point data)
BASE_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"


# 3. Define the enhanced function to fetch multi-variable data
def fetch_enhanced_climate(lat, lon, start, end):
    """
    Fetches Temperature (T2M), Precipitation (PRECTOTCORR), 
    and Soil Moisture (GWETPROF) from NASA POWER for specified coordinates.
    """
    params = {
        "parameters": PARAMETERS,
        "community": "AG",  # AG = Agroclimatology
        "format": "JSON",
        "latitude": lat,
        "longitude": lon,
        "start": start,
        "end": end,
    }

    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()  # Throws an exception if the request fails

    # Extract parameters dictionary directly
    param_data = response.json()["properties"]["parameter"]

    # Construct the multi-variable DataFrame safely from dictionary fields
    df = pd.DataFrame({
        "temp_celsius": param_data["T2M"],
        "precipitation_mm": param_data["PRECTOTCORR"],
        "soil_moisture": param_data["GWETPROF"]
    })

    # Clean the index and transform it into a column
    df.index = pd.to_datetime(df.index)
    df = df.sort_index().reset_index().rename(columns={"index": "date"})

    return df


# 4. Iterate through all locations and fetch temperature data
all_data = []

print("Starting to fetch temperature data from NASA POWER...")
print(f"Time Range: {START_DATE} to {END_DATE}")
print("-" * 50)

for loc in locations:
    name = loc["name"]
    lat = loc["lat"]
    lon = loc["lon"]

    print(f"Fetching data for: {name} (lat={lat}, lon={lon})")

    try:
        df_temp = fetch_enhanced_climate(lat, lon, START_DATE, END_DATE)
        df_temp["location"] = name
        df_temp["lat"] = lat
        df_temp["lon"] = lon

        all_data.append(df_temp)
        print(f"  ✓ Successfully fetched {len(df_temp)} records")

    except Exception as e:
        print(f"  ✗ Error: {e}")

    # Politeness delay to prevent making requests too quickly and getting rate-limited by the server
    time.sleep(1)

# 5. Merge all data
if all_data:
    combined_df = pd.concat(all_data, ignore_index=True)
    
    # Re-order columns to keep it intuitive and structured
    column_order = ["date", "location", "lat", "lon", "temp_celsius", "precipitation_mm", "soil_moisture"]
    combined_df = combined_df[column_order]
    print(f"✅ Successfully fetched and combined data for {len(locations)} locations.")
else:
    print("❌ Critical: No data records were fetched successfully.")

Starting to fetch temperature data from NASA POWER...
Time Range: 20200101 to 20261231
--------------------------------------------------
Fetching data for: Kigali City (lat=-1.9441, lon=30.0619)
  ✓ Successfully fetched 2437 records
Fetching data for: Eastern Province (lat=-1.95, lon=30.45)
  ✓ Successfully fetched 2437 records
Fetching data for: Western Province (lat=-2.15, lon=29.35)
  ✓ Successfully fetched 2437 records
Fetching data for: Northern Province (lat=-1.65, lon=29.85)
  ✓ Successfully fetched 2437 records
Fetching data for: Southern Province (lat=-2.45, lon=29.75)
  ✓ Successfully fetched 2437 records
✅ Successfully fetched and combined data for 5 locations.


In [3]:
combined_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12185 entries, 0 to 12184
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              12185 non-null  datetime64[us]
 1   location          12185 non-null  str           
 2   lat               12185 non-null  float64       
 3   lon               12185 non-null  float64       
 4   temp_celsius      12185 non-null  float64       
 5   precipitation_mm  12185 non-null  float64       
 6   soil_moisture     12185 non-null  float64       
dtypes: datetime64[us](1), float64(5), str(1)
memory usage: 666.5 KB


In [4]:
combined_df.head(5)

,date,location,lat,lon,temp_celsius,precipitation_mm,soil_moisture
0,2020-01-01,Kigali City,-1.9441,30.0619,20.34,1.66,0.83
1,2020-01-02,Kigali City,-1.9441,30.0619,20.19,1.62,0.82
2,2020-01-03,Kigali City,-1.9441,30.0619,20.09,0.76,0.82
3,2020-01-04,Kigali City,-1.9441,30.0619,19.30,29.49,0.83
4,2020-01-05,Kigali City,-1.9441,30.0619,19.55,2.49,0.83


In [5]:
combined_df["location"].value_counts()

location
Kigali City          2437
Eastern Province     2437
Western Province     2437
Northern Province    2437
Southern Province    2437
Name: count, dtype: int64

In [6]:
combined_df.isna().sum()

date                0
location            0
lat                 0
lon                 0
temp_celsius        0
precipitation_mm    0
soil_moisture       0
dtype: int64

In [7]:
# Save to CSV
output_file = "data/raw/rwanda_temperature_2020_2026.csv"
combined_df.to_csv(output_file, index=False)
print(f"✅ Data successfully saved to: {output_file}")
print(f"   Total Records: {len(combined_df)}")
print(f"   Unique Locations: {combined_df['location'].nunique()}")
print(f"   Date Range: {combined_df['date'].min()} to {combined_df['date'].max()}")

✅ Data successfully saved to: data/raw/rwanda_temperature_2020_2026.csv
   Total Records: 12185
   Unique Locations: 5
   Date Range: 2020-01-01 00:00:00 to 2026-09-02 00:00:00


# 2. USD/RWF Exchange Rates

In [8]:
import yfinance as yf

def fetch_exchange_rates(start_date="2019-12-01", end_date="2026-08-31"):
    """
    Pulls USD to RWF exchange rates and aggregates to a monthly average.
    """
    # RWF=X is the Yahoo Finance ticker for USD to Rwandan Franc
    forex_data = yf.download("RWF=X", start=start_date, end=end_date, progress=False)
    
    # Handle potential MultiIndex columns from newer yfinance versions
    close_prices = forex_data['Close']
    if isinstance(close_prices, pd.DataFrame):
        close_prices = close_prices.iloc[:, 0]
    
    # Isolate the closing price and resample to monthly mean
    df_forex = close_prices.resample('MS').mean().reset_index()
    df_forex.columns = ['date', 'usd_rwf_rate']
    
    # Shift date to the 15th to match WFP reporting dates exactly
    df_forex['date'] = df_forex['date'] + pd.DateOffset(days=14)
    
    return df_forex

df_exchange = fetch_exchange_rates()
df_exchange.head(5)

,date,usd_rwf_rate
0,2019-12-15,915.152055
1,2020-01-15,924.303552
2,2020-02-15,921.398273
3,2020-03-15,928.425052
4,2020-04-15,923.189237


In [9]:
df_exchange.info()

<class 'pandas.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype        
---  ------        --------------  -----        
 0   date          81 non-null     datetime64[s]
 1   usd_rwf_rate  81 non-null     float64      
dtypes: datetime64[s](1), float64(1)
memory usage: 1.4 KB


In [10]:
df_exchange.isnull().sum()

date            0
usd_rwf_rate    0
dtype: int64

In [11]:
# Sort chronologically to ensure shift() works backwards in time securely
df_exchange = df_exchange.sort_values('date')

# Forex Depreciation Rates
df_exchange['usd_rwf_mom_pct'] = df_exchange['usd_rwf_rate'].pct_change(1)

df_exchange.isnull().sum()

date               0
usd_rwf_rate       0
usd_rwf_mom_pct    1
dtype: int64

In [12]:
output_file = "data/raw/rwanda_exchange_2020_2026.csv"
df_exchange.to_csv(output_file, index=False)
print(f"✅ Data successfully saved to: {output_file}")
print(f"   Total Records: {len(df_exchange)}")
print(f"   Date Range: {df_exchange['date'].min()} to {df_exchange['date'].max()}")

✅ Data successfully saved to: data/raw/rwanda_exchange_2020_2026.csv
   Total Records: 81
   Date Range: 2019-12-15 00:00:00 to 2026-08-15 00:00:00


# 3. LOCAL CPI DATA (From provided NISR Excel)

In [13]:
def load_cpi_data(filepath="data/raw/CPI_time_series_July 2026.xls"):
    """
    Parses the Urban CPI sheet to extract General and Food CPI indices,
    formatting the timeline to match the 15th of each month.
    """
    df_urban = pd.read_excel(filepath, sheet_name='Urban')
    
    # Dates are located in row index 2, starting from column 5
    dates = df_urban.iloc[2, 5:].values
    
    # General Index is at row index 4, Food CPI is at row index 5
    general_cpi = df_urban.iloc[4, 5:].values
    food_cpi = df_urban.iloc[5, 5:].values
    
    # Construct a clean DataFrame
    df_cpi = pd.DataFrame({
        'date': pd.to_datetime(dates),
        'general_cpi': general_cpi,
        'food_cpi': food_cpi
    })
    
    # Drop NaNs and shift date to the 15th to match WFP reporting dates
    df_cpi = df_cpi.dropna()
    df_cpi['date'] = df_cpi['date'].apply(lambda x: x.replace(day=15))
    
    # Ensure numeric types
    df_cpi['general_cpi'] = pd.to_numeric(df_cpi['general_cpi'])
    df_cpi['food_cpi'] = pd.to_numeric(df_cpi['food_cpi'])
    
    return df_cpi

df_cpi = load_cpi_data("data/raw/CPI_time_series_July 2026.xls")
df_cpi.head(5)

,date,general_cpi,food_cpi
0,2009-02-15,81.387210,76.529674
1,2009-03-15,82.068220,78.055348
2,2009-04-15,81.934941,77.667427
3,2009-05-15,81.213300,76.065934
4,2009-06-15,80.838231,75.101367


In [14]:
df_cpi.info()

<class 'pandas.DataFrame'>
RangeIndex: 210 entries, 0 to 209
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         210 non-null    datetime64[us]
 1   general_cpi  210 non-null    float64       
 2   food_cpi     210 non-null    float64       
dtypes: datetime64[us](1), float64(2)
memory usage: 5.1 KB


In [15]:
df_cpi.isnull().sum()

date           0
general_cpi    0
food_cpi       0
dtype: int64

In [16]:
# 1. Ensure the date column is in datetime format and sort
df_cpi['date'] = pd.to_datetime(df_cpi['date'])
df_cpi = df_cpi.sort_values('date')

# 2. Filter for rows between January 1, 2020 and December 31, 2026
df_filtered = df_cpi[(df_cpi['date'] >= '2020-01-01') & (df_cpi['date'] <= '2026-12-31')].copy()

# 3. Calculate CPI Inflation Rates ON THE FILTERED DATAFRAME
df_filtered['cpi_mom_pct'] = df_filtered['general_cpi'].pct_change(1)
df_filtered['food_cpi_mom_pct'] = df_filtered['food_cpi'].pct_change(1)
df_filtered['cpi_yoy_pct'] = df_filtered['general_cpi'].pct_change(12)

# 4. Save the filtered DataFrame (which now correctly contains the new columns)
output_file = "data/raw/rwanda_cpi_2020_2026.csv"
df_filtered.to_csv(output_file, index=False)

# 5. Print verification
print(f"✅ Data successfully saved to: {output_file}")
print(f"Total Records: {len(df_filtered)}")
print(f"Columns saved: {df_filtered.columns.tolist()}")

✅ Data successfully saved to: data/raw/rwanda_cpi_2020_2026.csv
Total Records: 79
Columns saved: ['date', 'general_cpi', 'food_cpi', 'cpi_mom_pct', 'food_cpi_mom_pct', 'cpi_yoy_pct']
